# CUAD QLoRA fine-tune (Kaggle)

Fine-tunes Qwen3-4B-Instruct with Unsloth QLoRA on the CUAD training data.

**Notebook settings** (right-hand panel):
- Accelerator: **GPU T4 x2** (training uses one GPU)
- Internet: **On** (needs a phone-verified Kaggle account)
- Optional: add a secret `WANDB_API_KEY` (Add-ons → Secrets) to log to Weights & Biases

**How to run:**
1. Run interactively with `SMOKE_TEST = True` (~10 min). Check it trains without errors and read
   `est_hours_full_run` in the printed summary.
2. Set `SMOKE_TEST = False`, then **Save Version → Save & Run All (Commit)**. That runs in the background
   (up to 12 h, browser can be closed) and keeps everything in `/kaggle/working` as the version's output.
3. If the summary says `"finished": false`, the 12-hour limit was hit: create a new version, **Add Input →
   your previous version's output**, set `RESUME` to the last checkpoint path, and commit again.

In [ ]:
REPO_URL = "https://github.com/lakshy-606/cuad-qlora-vllm.git"  # your public GitHub repo
SMOKE_TEST = True  # first run: 20 steps to check everything works and measure speed
RESUME = None  # e.g. "/kaggle/input/<previous-version>/cuad-qlora-vllm/outputs/qwen3_4b_cuad_lora/checkpoints/checkpoint-600"

In [ ]:
def sh(cmd):
    """Run a shell command, streaming its output; raise if it fails."""
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
%cd /kaggle/working
sh(f"rm -rf cuad-qlora-vllm && git clone -q {REPO_URL}")
%cd /kaggle/working/cuad-qlora-vllm
sh("pip install -q unsloth")
sh("pip install -q --no-deps -e .")

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B logging on")
except Exception:
    print("No WANDB_API_KEY secret: logging to this notebook only")

In [ ]:
# Downloads CUAD (~18 MB) and rebuilds exactly the same splits and SFT data as locally (fixed seed).
sh("python scripts/prepare_data.py")

In [ ]:
cmd = "CUDA_VISIBLE_DEVICES=0 python scripts/train.py --config configs/train.yaml"
if SMOKE_TEST:
    cmd += " --max-steps 20"
if RESUME:
    cmd += f" --resume {RESUME}"
sh(cmd)

In [ ]:
# Keep the output small: raw CUAD can be re-downloaded. The adapter and checkpoints stay.
sh("rm -rf data/raw")
sh("cat outputs/qwen3_4b_cuad_lora/train_summary.json")
sh("du -sh outputs/qwen3_4b_cuad_lora/* 2>/dev/null")